# Figure 3: Evidence For Reference-Conditioned Modeling

- Figure / panels: `Fig3a` to `Fig3f`, with `Fig3g` exported as an optional reserve panel
- Inputs: resolved metrics files from `artifacts/results/<dataset>/...`, Systema baseline outputs, payload `pkl`
- Outputs: `artifacts/paper_figures/main/Fig3_ReferenceConditioning/*`
- Run order: run after the Fig2 benchmark notebook; this notebook focuses on nearest vs random and reference baselines
- Dataset selector: edit `SELECTED_DATASET_KEYS` in the parameter cell below
- Role: Main text


In [ ]:
from __future__ import annotations

import importlib
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / 'scripts').exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
if str(repo_root / 'src') not in sys.path:
    sys.path.insert(0, str(repo_root / 'src'))

from scripts.common.paper_plot_style import apply_gears_paper_style, style_axis as paper_style_axis
from scripts.trishift.analysis import baseline_panel, stratified_benchmark, systema_mechanism

apply_gears_paper_style(font_scale=1.0)
from scripts.trishift.analysis._result_adapter import load_payload_item, load_metrics_df, resolve_result
from trishift._utils import normalize_condition
importlib.reload(baseline_panel)
importlib.reload(stratified_benchmark)
importlib.reload(systema_mechanism)


In [ ]:
ALL_DATASET_SPECS = [
    {'dataset_key': 'dixit', 'dataset_label': 'Dixit', 'subgroup_filter': None},
    {'dataset_key': 'adamson', 'dataset_label': 'Adamson', 'subgroup_filter': None},
    {'dataset_key': 'norman', 'dataset_label': 'Norman', 'subgroup_filter': None},
    {'dataset_key': 'replogle_k562_essential', 'dataset_label': 'Replogle K562', 'subgroup_filter': None},
    {'dataset_key': 'replogle_rpe1_essential', 'dataset_label': 'Replogle RPE1', 'subgroup_filter': None},
]
AVAILABLE_DATASET_KEYS = [spec['dataset_key'] for spec in ALL_DATASET_SPECS]
SELECTED_DATASET_KEYS = ['adamson', 'norman']  # edit here
invalid_dataset_keys = [key for key in SELECTED_DATASET_KEYS if key not in AVAILABLE_DATASET_KEYS]
if invalid_dataset_keys:
    raise ValueError(f'Unknown dataset keys: {invalid_dataset_keys}. Available: {AVAILABLE_DATASET_KEYS}')
DATASET_SPECS = [spec for spec in ALL_DATASET_SPECS if spec['dataset_key'] in SELECTED_DATASET_KEYS]
if not DATASET_SPECS:
    raise ValueError('SELECTED_DATASET_KEYS produced an empty dataset list.')

REFERENCE_MODELS = ['trishift_nearest']  # edit here
DISTANCE_MODELS = ['trishift_nearest']  # edit here
GENERIC_SHIFT_MODELS = ['trishift_nearest']
NORMAN_SUBGROUP_MODELS = ['trishift_nearest', 'biolord', 'gears', 'genepert', 'scgpt']
AVAILABLE_SPLIT_IDS = [1, 2, 3, 4, 5]
SELECTED_SPLIT_IDS = AVAILABLE_SPLIT_IDS.copy()  # edit here
invalid_split_ids = [split_id for split_id in SELECTED_SPLIT_IDS if split_id not in AVAILABLE_SPLIT_IDS]
if invalid_split_ids:
    raise ValueError(f'Unknown split ids: {invalid_split_ids}. Available: {AVAILABLE_SPLIT_IDS}')
SPLIT_IDS = [int(split_id) for split_id in SELECTED_SPLIT_IDS]
RESULT_MODE = 'default'
OUT_ROOT = repo_root / 'artifacts' / 'paper_figures' / 'main' / 'Fig3_ReferenceConditioning'
OUT_ROOT.mkdir(parents=True, exist_ok=True)
REFERENCE_CASE_OVERRIDE = {'dataset_key': 'norman', 'condition': 'CITED1+ctrl', 'split_id': 4}
MODEL_LABELS = {
    'trishift_nearest': 'TriShift',
    'biolord': 'bioLORD',
    'gears': 'GEARS',
    'genepert': 'GenePert',
    'scgpt': 'scGPT',
    'systema_nonctl_mean': 'Perturbed mean',
    'systema_matching_mean': 'Matching mean',
    'Truth': 'Truth',
}
MODEL_COLORS = {
    'TriShift': '#9FD9D3',
    'bioLORD': '#EAC6A8',
    'GEARS': '#A7C7E7',
    'GenePert': '#C7B8EA',
    'scGPT': '#B8D8BA',
    'Perturbed mean': '#BFBFBF',
    'Matching mean': '#D9D9D9',
    'Truth': '#7F7F7F',
}
print('Selected datasets:', SELECTED_DATASET_KEYS)
print('Selected splits:', SPLIT_IDS)
print('Result mode:', RESULT_MODE)
print('Reference case override:', REFERENCE_CASE_OVERRIDE)
print('Norman subgroup models:', NORMAN_SUBGROUP_MODELS)


In [ ]:
def _read_csv_optional(path: Path) -> pd.DataFrame:
    return pd.read_csv(path) if path.exists() and path.stat().st_size > 5 else pd.DataFrame()


def _run_analysis_cli(module_name: str, *, dataset_key: str, models: list[str], subgroup_filter: str | None, out_dir: Path) -> None:
    cmd = [
        sys.executable,
        '-m',
        module_name,
        '--dataset',
        dataset_key,
        '--models',
        ','.join(models),
        '--split_ids',
        ','.join(map(str, SPLIT_IDS)),
        '--out_root',
        str(out_dir),
        '--result_mode',
        RESULT_MODE,
    ]
    if subgroup_filter:
        cmd.extend(['--subgroup_filter', subgroup_filter])
    if module_name.endswith('systema_mechanism'):
        cmd.extend(['--paths_path', str(repo_root / 'configs' / 'paths.yaml')])
    subprocess.run(cmd, cwd=repo_root, check=True, capture_output=True, text=True)


def safe_reference_benchmark(dataset_key: str, subgroup_filter: str | None = None) -> dict[str, object]:
    out_dir = OUT_ROOT / f'reference_{dataset_key}'
    out_dir.mkdir(parents=True, exist_ok=True)
    try:
        _run_analysis_cli(
            'scripts.trishift.analysis.baseline_panel',
            dataset_key=dataset_key,
            models=REFERENCE_MODELS,
            subgroup_filter=subgroup_filter,
            out_dir=out_dir,
        )
        return {
            'status': 'ready',
            'result': {
                'out_dir': out_dir,
                'summary_df': _read_csv_optional(out_dir / 'baseline_panel_summary.csv'),
                'raw_df': _read_csv_optional(out_dir / 'baseline_panel_raw.csv'),
                'ranking_df': _read_csv_optional(out_dir / 'baseline_panel_ranking.csv'),
            },
        }
    except Exception as exc:
        return {'status': 'pending', 'error': str(exc)}


def safe_mechanism_summary(dataset_key: str, subgroup_filter: str | None = None) -> dict[str, object]:
    out_dir = OUT_ROOT / f'mechanism_{dataset_key}'
    out_dir.mkdir(parents=True, exist_ok=True)
    try:
        _run_analysis_cli(
            'scripts.trishift.analysis.systema_mechanism',
            dataset_key=dataset_key,
            models=GENERIC_SHIFT_MODELS,
            subgroup_filter=subgroup_filter,
            out_dir=out_dir,
        )
        return {
            'status': 'ready',
            'result': {
                'out_dir': out_dir,
                'residual_summary_df': _read_csv_optional(out_dir / 'residualized_systema_summary.csv'),
                'centroid_summary_df': _read_csv_optional(out_dir / 'centroid_accuracy_summary.csv'),
                'difficulty_bin_df': _read_csv_optional(out_dir / 'difficulty_bin_generic_shift_summary.csv'),
            },
        }
    except Exception as exc:
        return {'status': 'pending', 'error': str(exc)}


def _style_axis(ax: plt.Axes, *, grid_axis: str = 'y') -> None:
    paper_style_axis(ax, grid_axis=grid_axis)
    ax.set_axisbelow(True)


def _shrink_bars(ax: plt.Axes, scale: float = 0.72) -> None:
    for patch in ax.patches:
        width = patch.get_width()
        if width <= 0:
            continue
        new_width = width * scale
        patch.set_x(patch.get_x() + (width - new_width) / 2)
        patch.set_width(new_width)


def render_grouped_bar(df: pd.DataFrame, metric_col: str, metric_label: str, out_path: Path, order: list[str] | None = None, hue_order: list[str] | None = None) -> None:
    fig, ax = plt.subplots(figsize=(8.4, 5.4), dpi=220)
    if df.empty:
        ax.text(0.5, 0.5, f'No rows for {metric_col}', ha='center', va='center'); ax.axis('off')
    else:
        work = df.copy()
        if order is not None and 'dataset_label' in work.columns:
            work['dataset_label'] = pd.Categorical(work['dataset_label'], categories=order, ordered=True)
            work = work.sort_values('dataset_label')
        present_hue_order = [label for label in hue_order if label in set(work['label'])] if hue_order is not None else None
        sns.barplot(data=work, x='dataset_label', y=metric_col, hue='label', hue_order=present_hue_order, palette=MODEL_COLORS, ax=ax, saturation=0.95, errorbar=None)
        _shrink_bars(ax)
        ax.set_xlabel('')
        ax.set_ylabel(metric_label)
        ax.set_title(metric_label)
        ax.tick_params(axis='x', rotation=32)
        ax.legend(title='', frameon=False, ncol=min(4, work['label'].nunique()), loc='upper left', bbox_to_anchor=(0.02, 1.02), borderaxespad=0.0, handlelength=0.9, handletextpad=0.35, columnspacing=0.8)
        for patch in ax.patches:
            patch.set_edgecolor('black'); patch.set_linewidth(0.5)
        _style_axis(ax)
    fig.tight_layout(); fig.savefig(out_path, bbox_inches='tight'); plt.close(fig)


def _build_reference_case(dataset_key: str, condition: str, split_id: int):
    condition_key = normalize_condition(condition)
    try:
        _, near_payload = load_payload_item(
            dataset=dataset_key,
            model_name='trishift_nearest',
            split_id=split_id,
            condition=None,
            result_mode=RESULT_MODE,
        )
    except Exception:
        return None, None
    near_item = {normalize_condition(k): v for k, v in near_payload.items()}.get(condition_key)
    if near_item is None:
        return None, None
    truth = np.asarray(near_item['Truth'], dtype=float)
    ctrl = np.asarray(near_item['Ctrl'], dtype=float)
    near_pred = np.asarray(near_item['Pred'], dtype=float)
    genes = np.asarray(near_item.get('DE_name', [f'g{i}' for i in range(near_pred.shape[1])])).astype(str)
    truth_delta = truth.mean(axis=0) - ctrl.mean(axis=0)
    order = np.argsort(-np.abs(truth_delta))[:12]
    rows = [
        pd.DataFrame({'Gene': genes[order], 'Expression': truth_delta[order], 'Group': 'Truth'}),
        pd.DataFrame({'Gene': genes[order], 'Expression': near_pred.mean(axis=0)[order] - ctrl.mean(axis=0)[order], 'Group': 'TriShift'}),
    ]
    return {'dataset_key': dataset_key, 'condition': condition_key, 'split_id': int(split_id)}, pd.concat(rows, ignore_index=True)


def pick_reference_case():
    if REFERENCE_CASE_OVERRIDE in (None, {}):
        raise ValueError('REFERENCE_CASE_OVERRIDE must be set explicitly')
    case_meta, case_df = _build_reference_case(
        dataset_key=REFERENCE_CASE_OVERRIDE['dataset_key'],
        condition=REFERENCE_CASE_OVERRIDE['condition'],
        split_id=int(REFERENCE_CASE_OVERRIDE['split_id']),
    )
    if case_meta is None or case_df is None:
        raise ValueError(f"Reference case could not be resolved: {REFERENCE_CASE_OVERRIDE}")
    return case_meta, case_df


In [ ]:
reference_runs, mechanism_runs = [], []
for spec in DATASET_SPECS:
    reference_runs.append({**spec, **safe_reference_benchmark(spec['dataset_key'], subgroup_filter=spec.get('subgroup_filter'))})
    mechanism_runs.append({**spec, **safe_mechanism_summary(spec['dataset_key'], subgroup_filter=spec.get('subgroup_filter'))})

reference_frames = []
for row in reference_runs:
    if row['status'] == 'ready':
        df = row['result']['summary_df'].copy()
        df['dataset_label'] = row['dataset_label']
        reference_frames.append(df)
reference_summary_df = pd.concat(reference_frames, ignore_index=True) if reference_frames else pd.DataFrame()
reference_summary_df.to_csv(OUT_ROOT / 'reference_summary_all.csv', index=False, encoding='utf-8-sig')

residual_frames, centroid_frames, bin_frames = [], [], []
for row in mechanism_runs:
    if row['status'] != 'ready':
        continue
    spec = row['dataset_label']
    residual_df = row['result']['residual_summary_df'].copy()
    centroid_df = row['result']['centroid_summary_df'].copy()
    bin_df = row['result']['difficulty_bin_df'].copy()
    if not residual_df.empty:
        residual_df['dataset_label'] = row['dataset_label']
        residual_frames.append(residual_df)
    if not centroid_df.empty:
        centroid_df['dataset_label'] = row['dataset_label']
        centroid_frames.append(centroid_df)
    if not bin_df.empty:
        bin_df['dataset_label'] = row['dataset_label']
        bin_frames.append(bin_df)
residual_summary_df = pd.concat(residual_frames, ignore_index=True) if residual_frames else pd.DataFrame()
centroid_summary_df = pd.concat(centroid_frames, ignore_index=True) if centroid_frames else pd.DataFrame()
difficulty_bin_df = pd.concat(bin_frames, ignore_index=True) if bin_frames else pd.DataFrame()
residual_summary_df.to_csv(OUT_ROOT / 'residualized_systema_summary_all.csv', index=False, encoding='utf-8-sig')
centroid_summary_df.to_csv(OUT_ROOT / 'centroid_accuracy_summary_all.csv', index=False, encoding='utf-8-sig')
difficulty_bin_df.to_csv(OUT_ROOT / 'difficulty_bin_generic_shift_summary_all.csv', index=False, encoding='utf-8-sig')
display(reference_summary_df.head())
display(residual_summary_df.head())
display(centroid_summary_df.head())

norman_subgroup_summary_df = pd.DataFrame()
norman_subgroup_df = pd.DataFrame()
if 'norman' in SELECTED_DATASET_KEYS:
    subgroup_dir = OUT_ROOT / 'norman_subgroup_all'
    subgroup_dir.mkdir(parents=True, exist_ok=True)
    try:
        _run_analysis_cli(
            'scripts.trishift.analysis.baseline_panel',
            dataset_key='norman',
            models=NORMAN_SUBGROUP_MODELS,
            subgroup_filter=None,
            out_dir=subgroup_dir,
        )
        norman_subgroup_summary_df = _read_csv_optional(subgroup_dir / 'baseline_panel_summary.csv')
        norman_subgroup_df = _read_csv_optional(subgroup_dir / 'norman_subgroup_panel.csv')
        if not norman_subgroup_summary_df.empty:
            norman_subgroup_summary_df['label'] = norman_subgroup_summary_df['model_name'].map(MODEL_LABELS).fillna(norman_subgroup_summary_df.get('label', norman_subgroup_summary_df['model_name']))
        if not norman_subgroup_df.empty:
            norman_subgroup_df['label'] = norman_subgroup_df['model_name'].map(MODEL_LABELS).fillna(norman_subgroup_df.get('label', norman_subgroup_df['model_name']))
    except Exception as exc:
        print(f'Norman subgroup benchmark pending: {exc}')
norman_subgroup_summary_df.to_csv(OUT_ROOT / 'fig3_norman_subgroup_summary.csv', index=False, encoding='utf-8-sig')
norman_subgroup_df.to_csv(OUT_ROOT / 'fig3_norman_subgroup_metrics.csv', index=False, encoding='utf-8-sig')
display(norman_subgroup_df.head())


In [ ]:
dataset_order = [spec['dataset_label'] for spec in DATASET_SPECS]
nearest_random_df = reference_summary_df[reference_summary_df['model_name'].isin(['trishift_nearest'])].copy()
if not nearest_random_df.empty:
    nearest_random_df['label'] = nearest_random_df['model_name'].map({'trishift_nearest': 'TriShift'})
render_grouped_bar(nearest_random_df, 'mean_pearson', 'Reference retrieval', OUT_ROOT / 'fig3a_nearest_vs_random.png', order=dataset_order)

systema_plot_df = reference_summary_df[reference_summary_df['model_name'].isin(REFERENCE_MODELS)].copy()
if not systema_plot_df.empty:
    systema_plot_df['label'] = systema_plot_df['model_name'].map(MODEL_LABELS).fillna(systema_plot_df['model_name'])
render_grouped_bar(systema_plot_df, 'mean_systema_corr_20de_allpert', 'Systema Pearson', OUT_ROOT / 'fig3b_systema_pearson.png', order=dataset_order)

signal_df = reference_summary_df[reference_summary_df['model_name'].isin(REFERENCE_MODELS)].copy()
metric_cols = ['mean_pearson', 'mean_nmse']
if not signal_df.empty:
    signal_df['label'] = signal_df['model_name'].map(MODEL_LABELS).fillna(signal_df['model_name'])
    ref_heatmap = signal_df[['label', 'dataset_label'] + [c for c in metric_cols if c in signal_df.columns]].copy().melt(id_vars=['label', 'dataset_label'], var_name='metric', value_name='value')
    ref_heatmap['metric'] = ref_heatmap['metric'].str.replace('mean_', '', regex=False)
    ref_heatmap['column'] = ref_heatmap['dataset_label'] + '\n' + ref_heatmap['metric']
    ref_table = ref_heatmap.pivot_table(index='label', columns='column', values='value', aggfunc='mean')
else:
    ref_table = pd.DataFrame()
fig, ax = plt.subplots(figsize=(max(9, ref_table.shape[1] * 1.0 if not ref_table.empty else 9), 4.8), dpi=220)
if ref_table.empty:
    ax.text(0.5, 0.5, 'No reference-baseline summary available', ha='center', va='center'); ax.axis('off')
else:
    sns.heatmap(ref_table, annot=False, cmap='viridis', linewidths=0.0, linecolor=None, ax=ax, cbar_kws={'shrink': 0.88, 'pad': 0.02})
    ax.set_title('TriShift vs reference baselines')
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.tick_params(axis='x', labelrotation=58, labelsize=8)
    ax.tick_params(axis='y', labelrotation=0, labelsize=9)
    for tick in ax.get_xticklabels():
        tick.set_ha('right')
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_edgecolor('black')
        spine.set_linewidth(1.0)
plt.tight_layout(); plt.savefig(OUT_ROOT / 'fig3c_reference_baselines.png', bbox_inches='tight'); plt.close()

fig3d_hue_order = ['TriShift', 'Perturbed mean']
residual_plot_df = residual_summary_df[residual_summary_df.get('model_name', pd.Series(dtype=str)).isin(REFERENCE_MODELS)].copy() if 'model_name' in residual_summary_df.columns else pd.DataFrame()
if not residual_plot_df.empty:
    residual_plot_df['label'] = residual_plot_df['model_name'].map(MODEL_LABELS).fillna(residual_plot_df['model_name'])
render_grouped_bar(
    residual_plot_df,
    'residualized_systema_corr_20de_allpert',
    'Residualized Systema Pearson',
    OUT_ROOT / 'fig3d_residualized_systema_pearson.png',
    order=dataset_order,
    hue_order=fig3d_hue_order,
)

fig3e_hue_order = ['TriShift', 'Perturbed mean']
centroid_plot_df = centroid_summary_df[centroid_summary_df.get('model_name', pd.Series(dtype=str)).isin(REFERENCE_MODELS)].copy() if 'model_name' in centroid_summary_df.columns else pd.DataFrame()
if not centroid_plot_df.empty:
    centroid_plot_df['label'] = centroid_plot_df['model_name'].map(MODEL_LABELS).fillna(centroid_plot_df['model_name'])
render_grouped_bar(
    centroid_plot_df,
    'centroid_accuracy',
    'Centroid accuracy',
    OUT_ROOT / 'fig3e_centroid_accuracy.png',
    order=dataset_order,
    hue_order=fig3e_hue_order,
)


In [ ]:
try:
    norman_metrics_df = load_metrics_df(resolve_result(dataset='norman', model_name='trishift_nearest', result_mode=RESULT_MODE))
    norman_schema_df = (
        norman_metrics_df[['condition', 'subgroup']]
        .drop_duplicates()
        .groupby('subgroup', as_index=False)
        .size()
        .rename(columns={'size': 'n_conditions'})
    )
except Exception:
    norman_schema_df = pd.DataFrame(columns=['subgroup', 'n_conditions'])

norman_schema_df.to_csv(OUT_ROOT / 'fig3_norman_subgroup_schema.csv', index=False, encoding='utf-8-sig')
fig, ax = plt.subplots(figsize=(4.8, 3.8), dpi=220)
if norman_schema_df.empty:
    ax.text(0.5, 0.5, 'Norman subgroup counts unavailable', ha='center', va='center')
    ax.axis('off')
else:
    sns.barplot(
        data=norman_schema_df,
        x='subgroup',
        y='n_conditions',
        order=['single', 'seen2', 'seen1', 'seen0'],
        color='#8FBBD9',
        ax=ax,
        errorbar=None,
        saturation=0.95,
    )
    _shrink_bars(ax, scale=0.76)
    ax.set_xlabel('')
    ax.set_ylabel('Number of conditions')
    ax.set_title('Norman subgroup split')
    _style_axis(ax, grid_axis='y')
    for patch in ax.patches:
        height = patch.get_height()
        if np.isfinite(height) and height > 0:
            ax.text(patch.get_x() + patch.get_width() / 2, height, f'{int(height)}', ha='center', va='bottom', fontsize=8)
fig.tight_layout()
fig.savefig(OUT_ROOT / 'fig3_norman_subgroup_schema.png', bbox_inches='tight')
plt.close(fig)

norman_plot_df = norman_subgroup_df[norman_subgroup_df.get('subgroup', pd.Series(dtype=str)).isin(['seen2', 'seen1', 'seen0'])].copy() if not norman_subgroup_df.empty else pd.DataFrame()
metrics_to_plot = [
    ('pearson', 'Pearson', 'fig3_norman_subgroup_pearson.png'),
    ('nmse', 'NMSE', 'fig3_norman_subgroup_nmse.png'),
    ('deg_mean_r2', 'DEG mean R2', 'fig3_norman_subgroup_deg_mean_r2.png'),
    ('systema_corr_20de_allpert', 'Systema Pearson', 'fig3_norman_subgroup_systema_pearson.png'),
]
for metric, metric_label, out_name in metrics_to_plot:
    fig, ax = plt.subplots(figsize=(6.2, 4.6), dpi=220)
    value_col = f'mean_{metric}'
    if norman_plot_df.empty or value_col not in norman_plot_df.columns:
        ax.text(0.5, 0.5, f'No rows for {metric}', ha='center', va='center')
        ax.axis('off')
    else:
        plot_df = norman_plot_df.copy()
        plot_df['label'] = plot_df['model_name'].map(MODEL_LABELS).fillna(plot_df['label'] if 'label' in plot_df.columns else plot_df['model_name'])
        sns.barplot(
            data=plot_df,
            x='subgroup',
            y=value_col,
            hue='label',
            order=['seen2', 'seen1', 'seen0'],
            hue_order=[MODEL_LABELS[m] for m in NORMAN_SUBGROUP_MODELS if MODEL_LABELS[m] in set(plot_df['label'])],
            palette=MODEL_COLORS,
            ax=ax,
            errorbar=None,
            saturation=0.95,
        )
        _shrink_bars(ax, scale=0.72)
        ax.set_xlabel('')
        ax.set_ylabel(metric_label)
        ax.set_title(metric_label)
        ax.tick_params(axis='x', rotation=18)
        for patch in ax.patches:
            patch.set_edgecolor('white')
            patch.set_linewidth(0.6)
        handles, labels = ax.get_legend_handles_labels()
        if ax.legend_ is not None:
            ax.legend_.remove()
        fig.legend(handles, labels, frameon=False, title='', ncol=min(5, len(labels)), loc='upper center', bbox_to_anchor=(0.5, 1.01), handlelength=0.9, handletextpad=0.35, columnspacing=0.8)
        _style_axis(ax, grid_axis='y')
    fig.tight_layout(rect=[0.0, 0.0, 1.0, 0.88])
    fig.savefig(OUT_ROOT / out_name, bbox_inches='tight')
    plt.close(fig)

display(norman_plot_df.head())


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14.8, 4.9), dpi=220)
plot_specs = [
    ('systema_corr_20de_allpert', 'Systema Pearson'),
    ('residualized_systema_corr_20de_allpert', 'Residualized Systema Pearson'),
    ('generic_projection_ratio', 'Generic-shift projection ratio'),
]
required_cols = {'model_name', 'dataset_label', 'train_distance_bin'}
if difficulty_bin_df.empty or not required_cols.issubset(difficulty_bin_df.columns):
    for ax in axes:
        ax.text(0.5, 0.5, 'No difficulty-bin summary available', ha='center', va='center')
        ax.axis('off')
else:
    from matplotlib.lines import Line2D

    work = difficulty_bin_df[difficulty_bin_df['model_name'].isin(GENERIC_SHIFT_MODELS)].copy()
    work['label'] = work['model_name'].map(MODEL_LABELS).fillna(work['model_name'])
    work['train_distance_bin'] = pd.Categorical(work['train_distance_bin'], categories=['near', 'medium', 'far'], ordered=True)
    dataset_style = {'Adamson': ('o', '-'), 'Norman': ('X', '--')}
    ordered_labels = [MODEL_LABELS[m] for m in GENERIC_SHIFT_MODELS if m in set(work['model_name'])]
    for ax, (metric_col, title) in zip(axes, plot_specs):
        if metric_col not in work.columns:
            ax.text(0.5, 0.5, f'Missing {metric_col}', ha='center', va='center')
            ax.axis('off')
            continue
        for label in ordered_labels:
            for dataset_label in dataset_order:
                sub = work[(work['label'] == label) & (work['dataset_label'] == dataset_label)].copy()
                if sub.empty:
                    continue
                marker, linestyle = dataset_style.get(dataset_label, ('o', '-'))
                sns.lineplot(
                    data=sub,
                    x='train_distance_bin',
                    y=metric_col,
                    sort=False,
                    marker=marker,
                    linestyle=linestyle,
                    color=MODEL_COLORS[label],
                    linewidth=2.0,
                    markersize=7,
                    legend=False,
                    ax=ax,
                )
        ax.set_xlabel('Train-distance bin')
        ax.set_ylabel(title)
        ax.set_title(title)
        _style_axis(ax)
    model_handles = [Line2D([0], [0], color=MODEL_COLORS[label], lw=2.2, label=label) for label in ordered_labels]
    dataset_handles = [
        Line2D([0], [0], color='#444444', lw=1.8, marker='o', linestyle='-', label='Adamson'),
        Line2D([0], [0], color='#444444', lw=1.8, marker='X', linestyle='--', label='Norman'),
    ]
    leg1 = axes[0].legend(handles=model_handles, title='', frameon=False, loc='upper left', bbox_to_anchor=(0.0, 1.22), ncol=min(3, len(model_handles)), borderaxespad=0.0)
    axes[0].add_artist(leg1)
    axes[-1].legend(handles=dataset_handles, title='', frameon=False, loc='upper right', bbox_to_anchor=(1.0, 1.22), ncol=2, borderaxespad=0.0)
    fig.text(0.5, 0.01, 'Note: In the single-perturbation setting, Matching mean is omitted because it is identical to Perturbed mean.', ha='center', va='bottom', fontsize=9, color='#555555')
fig.tight_layout(rect=[0.0, 0.04, 1.0, 0.89]); plt.savefig(OUT_ROOT / 'fig3g_generic_shift_dependence.png', bbox_inches='tight'); plt.close()
display(difficulty_bin_df.head())



In [ ]:
case_meta, case_df = pick_reference_case()
if case_meta is not None and case_df is not None:
    case_df.to_csv(OUT_ROOT / 'fig3f_reference_case_values.csv', index=False, encoding='utf-8-sig')
    plt.figure(figsize=(11.0, 5.8), dpi=220)
    ax = sns.barplot(data=case_df, x='Gene', y='Expression', hue='Group', palette=MODEL_COLORS, errorbar=None, saturation=0.95)
    _shrink_bars(ax, scale=0.76)
    for patch in ax.patches:
        patch.set_edgecolor('white'); patch.set_linewidth(0.7)
    ax.set_xlabel('')
    ax.set_ylabel('Expression change over control')
    ax.set_title(f"Reference-sensitive case: {case_meta['condition']} (split {case_meta['split_id']})")
    ax.tick_params(axis='x', rotation=32)
    ax.legend(title='', frameon=False, loc='upper center', bbox_to_anchor=(0.5, 1.18), ncol=min(3, case_df['Group'].nunique()))
    ax.axhline(y=0, color='#4A4A4A', linewidth=0.8)
    _style_axis(ax)
    plt.tight_layout(); plt.savefig(OUT_ROOT / 'fig3f_reference_case.png', bbox_inches='tight'); plt.close()
    print(case_meta); display(case_df.head(20))
print(OUT_ROOT)

